# Autonomous Master Manuscript Processing & Packaging Pipeline
### Turnkey Batch Pipeline: Reads All Books from `pdf/` and Generates Submission Deliverables in `output/`
**Course**: CSC15006 - Introduction to Natural Language Processing (NLP)  
**Official Standard**: `guide/Quy_định_đồ_án.pdf` (FIT HCMUS)  

---

## 1. Regulatory Specifications for Deliverable Folders (`guide/Quy_định_đồ_án.pdf`)

- **`ch/` (Mandatory)**: Thư mục chứa **CÁC TRANG HÁN NÔM**.
  - Tên ảnh: `<Tên sách>.<Số thứ tự hình ảnh>.jpg` (Độ phân giải cao 300 DPI).
  - `Label.txt`: Định dạng PPOCRLabel, mỗi dòng là một trang ảnh Hán Nôm và mảng JSON các hộp bao 4 điểm polygon `[[x1,y1],[x2,y2],[x3,y3],[x4,y4]]` bao quanh các dòng chữ Hán Nôm.
  - `fileState.txt`: Trạng thái đánh nhãn PPOCRLabel (`1` cho trang có nhãn, `0` cho trang chưa gán).
- **`vi/` (Mandatory)**: Thư mục chứa **CÁC TRANG QUỐC NGỮ**.
  - Chỉ chứa các trang có chữ Quốc ngữ (với sách song ngữ xen dòng T2b/T2bw, ảnh trang nội dung được phân bổ vào cả `ch/` và `vi/`, trong đó `ch/Label.txt` chỉ chứa box chữ Nôm, `vi/Label.txt` chỉ chứa box Quốc ngữ; với sách T2/T2w phân trang xen kẽ hoặc Tnom chỉ có Nôm, các trang được phân chia độc lập chính xác).
  - `Label.txt`: Hộp bao 4 điểm polygon bao quanh các dòng chữ Quốc ngữ.
  - `fileState.txt`: Trạng thái đánh nhãn tương ứng.
- **`misc/`**: Thư mục chứa **CÁC TRANG CÒN LẠI** (Trang bìa, trang giới thiệu số hóa thư viện Yale, mục lục, trang trắng, trang kết).
- **Nguyên tắc bảo toàn 100% số trang**: Hợp các trang trong `(ch ∪ vi ∪ misc)` phải bằng chính xác 100% số trang gốc của file PDF.
- **`<Tên sách>.pdf`**: Nhân bản nguyên vẹn file PDF tài liệu gốc.
- **`<Tên sách>.csv`**: Bảng song ngữ đã dóng hàng UTF-8 có BOM (4 cột chuẩn `img_id, sentence_id, SinoNom_Char, ChuQN_txt`).
- **`<Tên sách>.xlsx`**: Bảng song ngữ đã dóng hàng, định dạng màu ARGB (Chữ đen `FF000000` khớp âm, Chữ đỏ `FFFF0000` lệch âm).

**Thực thi 1 nút bấm**: Bấm **Kernel -> Restart Kernel and Run All Cells** để tự động xử lý và kiểm định toàn bộ sách.


In [ ]:
# Environment Setup and Core Module Imports
import os
import sys
import re
import time
import json
import shutil
import unicodedata
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

import cv2
import numpy as np
import pandas as pd
import openpyxl
import pymupdf

# Set UTF-8 encoding for standard output streams on Windows
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

# Ensure project root is in sys.path
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from core.config import (
    BASE_DIR, OUTPUT_DIR, DATASETS_MIDTERM_DIR, MIN_SIMILARITY_THRESHOLD,
    COLOR_BLACK, COLOR_RED, KIMHANNOM_BASE_URL
)
from core.classifier import BookClassifier
from core.preprocess import DocumentPreprocessor, cv2_imread_unicode
from core.cleaner import TextCleaner, PPOCRLabelFormatter
from core.dictionary import HanNomDictionary
from core.aligner import SentenceAligner
from core.evaluator import AlignmentEvaluator
from core.exporter import CorpusExporter

PDF_DIR = DATASETS_MIDTERM_DIR / "pdf"
CSV_DIR = DATASETS_MIDTERM_DIR / "csv"
XLSX_DIR = DATASETS_MIDTERM_DIR / "xlsx"

lexicon_size = len(HanNomDictionary.get_instance().char_to_readings)

print("=" * 80)
print("AUTONOMOUS MASTER PIPELINE INITIALIZED")
print("=" * 80)
print(f"Manuscript PDF Directory : {PDF_DIR}")
print(f"Deliverables Output Dir  : {OUTPUT_DIR}")
print(f"Phonetic Lexicon Entries : {lexicon_size} loaded")
print("Core modules loaded successfully.")
print("=" * 80)


In [ ]:
# Scan PDF Folder and Discover All Manuscript Entities
classifier = BookClassifier()
all_pdf_files = sorted(list(PDF_DIR.glob("*.pdf")))
txt_companion = PDF_DIR / "Thụ Dạ Lữ Hoài Ngâm Khúc_Dịch.txt"

catalog = []
for idx, pdf_path in enumerate(all_pdf_files, 1):
    stem = pdf_path.stem
    doc = pymupdf.open(str(pdf_path))
    pcount = len(doc)
    doc.close()
    
    csv_path = CSV_DIR / f"{stem}.csv"
    xlsx_path = XLSX_DIR / f"{stem}.xlsx"
    cls_res = classifier.classify(pdf_path)
    
    catalog.append({
        "STT": idx,
        "Book_Key": stem,
        "PDF_Filename": pdf_path.name,
        "Total_Pages": pcount,
        "Layout_Code": cls_res["book_type"],
        "Layout_Name": cls_res["type_name"],
        "Confidence": f"{cls_res['confidence'] * 100:.0f}%",
        "PDF_Path": pdf_path,
        "CSV_Path": csv_path if csv_path.exists() else None,
        "XLSX_Path": xlsx_path if xlsx_path.exists() else None,
    })

print("=" * 95)
print(f"DISCOVERED {len(catalog)} MANUSCRIPT ENTITIES IN {PDF_DIR.name}/")
if txt_companion.exists():
    print(f"Companion Translation Detected: {txt_companion.name}")
print("=" * 95)
print(f"{'STT':<4} | {'Code':<6} | {'Pages':<5} | {'Conf':<6} | {'Book Title':<50}")
print("=" * 95)
total_all_pages = sum(b['Total_Pages'] for b in catalog)
for b in catalog:
    print(f"{b['STT']:<4} | {b['Layout_Code']:<6} | {b['Total_Pages']:<5} | {b['Confidence']:<6} | {b['Book_Key'][:50]:<50}")
print("=" * 95)
print(f"TOTAL MANUSCRIPTS: {len(catalog)} | TOTAL PAGES ACROSS CATALOG: {total_all_pages}")
print("=" * 95)


In [ ]:
# Master Pipeline Execution Engine with Script-Specific ch/ and vi/ Routing
t_start_all = time.time()
execution_results = []

print("=" * 80)
print(f"STARTING PIPELINE EXECUTION FOR ALL {len(catalog)} MANUSCRIPTS")
print("=" * 80)

for b_idx, book_info in enumerate(catalog, 1):
    stem = book_info["Book_Key"]
    pdf_path = book_info["PDF_Path"]
    csv_path = book_info["CSV_Path"]
    layout_code = book_info["Layout_Code"]
    t0 = time.time()
    
    print(f"\n[{b_idx:02d}/{len(catalog):02d}] Processing: {stem}")
    print(f"  Layout: {layout_code} | Total Pages: {book_info['Total_Pages']}")
    
    b_dict = {
        "title": stem,
        "pdf_filename": pdf_path.name,
        "csv_filename": csv_path.name if csv_path else f"{stem}.csv",
        "layout_type": layout_code
    }
    
    # Output directories
    book_out_dir = OUTPUT_DIR / stem
    dir_ch = book_out_dir / "ch"
    dir_vi = book_out_dir / "vi"
    dir_misc = book_out_dir / "misc"
    
    # Clean any stale previous output
    if book_out_dir.exists():
        shutil.rmtree(book_out_dir)
    dir_ch.mkdir(parents=True, exist_ok=True)
    dir_vi.mkdir(parents=True, exist_ok=True)
    dir_misc.mkdir(parents=True, exist_ok=True)
    
    # Step 1: Render all pages at 300 DPI
    prep = DocumentPreprocessor(b_dict, OUTPUT_DIR)
    extracted_pages = prep.split_and_extract_images(str(pdf_path))
    
    # Step 2: Determine script-specific page assignments according to Quy định đồ án
    ch_img_names = set()
    vi_img_names = set()
    
    if csv_path and csv_path.exists():
        df_csv = pd.read_csv(csv_path)
        if "target_folder" in df_csv.columns:
            for _, r in df_csv.iterrows():
                tgt = str(r.get("target_folder", "")).strip()
                img = str(r.get("img_id", "")).strip()
                if img and img != "nan":
                    if "ch" in tgt: ch_img_names.add(img)
                    if "vi" in tgt: vi_img_names.add(img)
        else:
            # Default T2b/T2bw: content pages contain both Hán Nôm and Quốc ngữ
            for img in df_csv["img_id"].dropna().unique():
                img_str = str(img).strip()
                ch_img_names.add(img_str)
                vi_img_names.add(img_str)
                
    ch_pages, vi_pages, misc_pages = [], [], []
    for pno, name, img_p in extracted_pages:
        has_ch = name in ch_img_names
        has_vi = name in vi_img_names
        
        if has_ch:
            ch_pages.append((pno, name, img_p))
            shutil.copy2(str(img_p), str(dir_ch / name))
        if has_vi:
            vi_pages.append((pno, name, img_p))
            shutil.copy2(str(img_p), str(dir_vi / name))
            
        if not has_ch and not has_vi:
            misc_pages.append((pno, name, img_p))
            shutil.copy2(str(img_p), str(dir_misc / name))
            
    # Step 3: Replicate Source PDF to Deliverables Folder
    dest_pdf = book_out_dir / f"{stem}.pdf"
    shutil.copy2(str(pdf_path), str(dest_pdf))
    
    # Step 4: Sentence Alignment & Reconstruction
    aligner = SentenceAligner(b_dict, csv_path) if csv_path else None
    recs = aligner.align_book_data() if aligner else []
    
    # Step 5: Segment Bounding Polygons & Generate PPOCRLabel Annotations
    page_map = {}
    for r in recs:
        img_id = r["img_id"]
        if img_id not in page_map:
            page_map[img_id] = []
        page_map[img_id].append(r)
        
    is_vertical_nom = "Tnom" in layout_code
    ch_map = {}
    for pno, img_name, img_p in ch_pages:
        if is_vertical_nom:
            ch_boxes = prep.segment_vertical_nom_columns(img_p)
        else:
            seg = prep.segment_t2b_lines(img_p)
            ch_boxes = seg["ch"]
        page_recs = page_map.get(img_name, [])
        ch_map[img_name] = [
            {"transcription": page_recs[i]["SinoNom_Char"] if i < len(page_recs) else "", "points": b["points"]}
            for i, b in enumerate(ch_boxes)
        ]
        
    vi_map = {}
    for pno, img_name, img_p in vi_pages:
        seg = prep.segment_t2b_lines(img_p)
        page_recs = page_map.get(img_name, [])
        vi_boxes = seg["vi"]
        vi_map[img_name] = [
            {"transcription": page_recs[i]["ChuQN_txt"] if i < len(page_recs) else "", "points": b["points"]}
            for i, b in enumerate(vi_boxes)
        ]
        
    PPOCRLabelFormatter.write_ppocr_dataset(dir_ch, "ch", ch_map)
    PPOCRLabelFormatter.write_ppocr_dataset(dir_vi, "vi", vi_map)
    
    # Step 6: Quality & Similarity Evaluation
    evaluator = AlignmentEvaluator()
    eval_rep = evaluator.evaluate_dataset(recs)
    
    # Step 7: Export CSV (UTF-8 BOM) and Styled XLSX (ARGB Colors)
    out_csv = book_out_dir / f"{stem}.csv"
    out_xlsx = book_out_dir / f"{stem}.xlsx"
    CorpusExporter.export_csv(recs, out_csv)
    CorpusExporter.export_xlsx(recs, eval_rep["row_evaluations"], out_xlsx)
    
    elapsed = time.time() - t0
    print(f"  Done in {elapsed:.2f}s | ch: {len(ch_pages)} | vi: {len(vi_pages)} | misc: {len(misc_pages)} | Pairs: {len(recs)} | Sim: {eval_rep['similarity_percentage']:.2f}%")
    
    execution_results.append({
        "book": stem,
        "layout": layout_code,
        "total_pages": len(extracted_pages),
        "ch_pages": len(ch_pages),
        "vi_pages": len(vi_pages),
        "misc_pages": len(misc_pages),
        "aligned_pairs": len(recs),
        "similarity": eval_rep["similarity_percentage"],
        "passed": eval_rep["is_passed"],
        "elapsed_sec": round(elapsed, 2),
        "out_dir": book_out_dir,
        "pdf_path": dest_pdf,
        "csv_path": out_csv,
        "xlsx_path": out_xlsx,
        "ch_dir": dir_ch,
        "vi_dir": dir_vi,
        "misc_dir": dir_misc
    })

total_elapsed = time.time() - t_start_all
print("\n" + "=" * 80)
print(f"ALL {len(catalog)} BOOKS PROCESSED SUCCESSFULLY IN {total_elapsed:.2f} SECONDS ({total_elapsed/60:.1f} MINUTES)")
print("=" * 80)


In [ ]:
# Automated 10/10 Compliance Verification for All Deliverable Folders
def validate_ppocr_format(label_path: Path) -> bool:
    if not label_path.exists():
        return False
    lines = [l.strip() for l in label_path.read_text(encoding="utf-8").splitlines() if l.strip()]
    if not lines:
        return True # Valid empty label file when 0 images present for that script
    for line in lines:
        parts = line.split("\t")
        if len(parts) != 2:
            return False
        try:
            boxes = json.loads(parts[1])
            for b in boxes:
                if len(b.get("points", [])) != 4:
                    return False
        except Exception:
            return False
    return True

audit_results = []
print("=" * 80)
print("AUDITING DELIVERABLES COMPLIANCE (10/10 CRITERIA) ACROSS ALL BOOKS")
print("=" * 80)

for res in execution_results:
    stem = res["book"]
    out_dir = res["out_dir"]
    ch_dir = res["ch_dir"]
    vi_dir = res["vi_dir"]
    misc_dir = res["misc_dir"]
    csv_file = res["csv_path"]
    xlsx_file = res["xlsx_path"]
    pdf_file = res["pdf_path"]
    
    checks = {}
    
    # 1. 100% Page conservation across unique union of (ch ∪ vi ∪ misc)
    ch_img_set = set(p.name for p in ch_dir.glob("*.jpg"))
    vi_img_set = set(p.name for p in vi_dir.glob("*.jpg"))
    misc_img_set = set(p.name for p in misc_dir.glob("*.jpg"))
    total_preserved = len(ch_img_set.union(vi_img_set).union(misc_img_set))
    checks["1_Page_Conservation"] = total_preserved == res["total_pages"]
    
    # 2. Subdirectory ch/ (Mandatory)
    checks["2_Subdir_ch"] = (ch_dir / "Label.txt").exists() and (ch_dir / "fileState.txt").exists()
    
    # 3. Subdirectory vi/ (Mandatory)
    checks["3_Subdir_vi"] = (vi_dir / "Label.txt").exists() and (vi_dir / "fileState.txt").exists()
    
    # 4. Subdirectory misc/ (Present)
    checks["4_Subdir_misc"] = misc_dir.is_dir()
    
    # 5. Original PDF replicated
    checks["5_Replicated_PDF"] = pdf_file.exists() and pdf_file.stat().st_size > 0
    
    # 6. ch/Label.txt format
    checks["6_ch_PPOCRLabel_4pt"] = validate_ppocr_format(ch_dir / "Label.txt")
    
    # 7. vi/Label.txt format
    checks["7_vi_PPOCRLabel_4pt"] = validate_ppocr_format(vi_dir / "Label.txt")
    
    # 8. CSV UTF-8 with BOM format
    try:
        df_csv = pd.read_csv(csv_file, encoding="utf-8-sig")
        checks["8_CSV_BOM_4Cols"] = list(df_csv.columns[:4]) == ["img_id", "sentence_id", "SinoNom_Char", "ChuQN_txt"]
    except Exception:
        checks["8_CSV_BOM_4Cols"] = False
        
    # 9. XLSX ARGB font styling
    try:
        wb = openpyxl.load_workbook(xlsx_file)
        ws = wb.active
        checks["9_XLSX_ARGB_Colors"] = any(c.font and c.font.color and c.font.color.rgb for row in ws.iter_rows(min_row=2, max_row=10) for c in row)
    except Exception:
        checks["9_XLSX_ARGB_Colors"] = False
        
    # 10. Similarity metric >= 95.0%
    checks["10_Phonetic_Sim_95"] = res["similarity"] >= 95.0
    
    pass_count = sum(1 for v in checks.values() if v)
    all_ok = pass_count == 10
    
    audit_results.append({
        "Book_Title": stem,
        "Total_Pages": res["total_pages"],
        "ch_Pages": res["ch_pages"],
        "vi_Pages": res["vi_pages"],
        "misc_Pages": res["misc_pages"],
        "Aligned_Pairs": res["aligned_pairs"],
        "Similarity_%": f"{res['similarity']:.2f}%",
        "Audit_Score": f"{pass_count}/10",
        "Compliance": "PASS [10/10]" if all_ok else f"FAIL [{pass_count}/10]",
        "Runtime_s": f"{res['elapsed_sec']:.1f}s"
    })
    
    print(f"Book: {stem[:40]:<40} | ch:{res['ch_pages']:>3} | vi:{res['vi_pages']:>3} | misc:{res['misc_pages']:>3} | Audit: {pass_count}/10 | {'PASS' if all_ok else 'FAIL'}")

print("=" * 80)

# Final Deliverables Audit Scorecard
scorecard_df = pd.DataFrame(audit_results)
print("\n" + "=" * 115)
print("FINAL DELIVERABLES AUDIT SCORECARD")
print("=" * 115)
print(scorecard_df.to_string(index=False))
print("=" * 115)

all_passed = all(r["Compliance"] == "PASS [10/10]" for r in audit_results)
if all_passed:
    print("\nOVERALL SYSTEM STATUS: ALL MANUSCRIPTS PASSED 10/10 COMPLIANCE AUDIT")
    print(f"Deliverables folder is fully verified and ready for submission: {OUTPUT_DIR}")
else:
    failed_books = [r['Book_Title'] for r in audit_results if r['Compliance'] != 'PASS [10/10]']
    print(f"\nWARNING: {len(failed_books)} book(s) require attention: {failed_books}")


## 2. Deliverables Structure Verification

According to **`guide/Quy_định_đồ_án.pdf` (Page 4)**, the output folder is structured as follows:

```text
output/
|-- <Tên sách>/
|   |-- ch/
|   |   |-- <Tên sách>.<Số thứ tự hình ảnh>.jpg (Chỉ chứa các trang Hán Nôm)
|   |   |-- Label.txt        (PPOCRLabel 4-point polygon format)
|   |   `-- fileState.txt    (PPOCRLabel status 1/0)
|   |-- vi/
|   |   |-- <Tên sách>.<Số thứ tự hình ảnh>.jpg (Chỉ chứa các trang Quốc ngữ)
|   |   |-- Label.txt        (PPOCRLabel 4-point polygon format)
|   |   `-- fileState.txt    (PPOCRLabel status 1/0)
|   |-- misc/
|   |   `-- <Tên sách>.<Số thứ tự hình ảnh>.jpg (Trang bìa, mục lục, trang trắng, kết thúc)
|   |-- <Tên sách>.pdf       (Original PDF replicated)
|   |-- <Tên sách>.xlsx      (Bilingual parallel corpus, ARGB Black/Red styled fonts)
|   `-- <Tên sách>.csv       (Bilingual parallel corpus, UTF-8 with BOM)
```

> [!NOTE]
> To submit, compress the target book folders in `output/` into a standard `.zip` archive as specified by course instructors.
